# NFL: fit, compare and predict

Fits both model families on nflverse data, compares them on the validation
season, predicts the next week and builds neutral-site power ratings.
The CLI equivalent is `sportsml nfl download && sportsml nfl process && sportsml nfl fit sklearn`.

In [ ]:
import pandas as pd
import torch

from sportsml.core.process import process
from sportsml.inference import all_matchups, long_to_wide, upcoming_matchups
from sportsml.models.gnn import default_model, default_trainer, fit_gnn
from sportsml.models.sklearn import fit_sklearn
from sportsml.nfl import SPORT

Schedules and weekly team stats for every season since 1999 (about 20 seconds). Upcoming games are kept, with a missing `result`.

In [ ]:
games = process("nfl", SPORT.download())
games[["season", "week", "team", "opp_team", "location", "result"]].tail()

## Fit

Default split: the latest season is the test set, the one before is the
validation set (early stopping, win-probability calibration), the rest is
training. Each fit prints metrics per split.

In [ ]:
sk = fit_sklearn(games, "nfl", save_dir="../models/nfl/sklearn", overwrite=True)

In [ ]:
torch.manual_seed(0)
gnn = fit_gnn(
    games,
    "nfl",
    model=default_model("nfl"),
    trainer=default_trainer(),
    save_dir="../models/nfl/gnn",
    overwrite=True,
)

In [ ]:
pd.DataFrame(
    {name: res["metrics"]["val"] for name, res in [("sklearn", sk), ("gnn", gnn)]}
).T.round(4)

## Next week

One row per upcoming game, from the home team's perspective.

In [ ]:
upcoming = games[games.result.isna() & (games.season == games.season.max())]
week = upcoming_matchups(games, sk["predictor"], date=upcoming.week.min())
week[["week", "team", "opp_team"]].assign(
    sklearn=sk["predictor"].predict(week).win_prob.round(3),
    gnn=gnn["predictor"].predict(week).win_prob.round(3),
)

## Power ratings

Average predicted margin and win probability against every other team at a neutral site.

In [ ]:
predictor = gnn["predictor"]
pairs = predictor.predict(all_matchups(predictor))
pairs.groupby("team")[["pred_margin", "win_prob"]].mean().sort_values(
    "pred_margin", ascending=False
).round(3)

In [ ]:
long_to_wide(pairs, opp="opp_team").round(2)